
# Medical Appointment No Shows — Data Cleaning

## Tujuan

Notebook ini digunakan untuk melakukan **data cleaning dan quality checking** pada dataset *Medical Appointment No Shows* sebelum data dianalisis menggunakan SQL dan Tableau.

Fokus pembersihan:

1. Memuat dan memahami struktur dataset.
2. Menstandarkan nama kolom.
3. Memeriksa missing values dan duplikasi.
4. Memastikan format kolom tanggal sesuai.
5. Menangani nilai usia yang tidak valid.
6. Menghitung dan memvalidasi `waiting_days`.
7. Membuat fitur analisis seperti `age_group` dan `no_show_binary`.
8. Melakukan validasi akhir dan menyimpan dataset bersih.

> **Catatan:** Nilai yang tidak valid tidak langsung dihapus dari baris data. Jika memungkinkan, nilai tersebut ditandai dan diubah menjadi `NaN` agar jumlah observasi tetap terjaga dan proses cleaning dapat diaudit.


## 1. Import Library dan Pengaturan Tampilan

In [30]:

import os
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

# Menampilkan seluruh kolom ketika DataFrame ditampilkan
pd.set_option("display.max_columns", None)

# Membatasi tampilan angka desimal
pd.set_option("display.float_format", "{:.2f}".format)



## 2. Memuat Dataset

Sesuaikan `RAW_DATA_PATH` dengan lokasi file CSV pada komputer atau Google Drive.

Dataset asli disimpan dalam variabel `df_raw` agar tetap tersedia sebagai referensi. Salinan kerja untuk proses cleaning akan disimpan dalam `df_clean`.


In [31]:

# Sesuaikan path berikut dengan lokasi dataset raw milik Anda
RAW_DATA_PATH = Path(
    "/content/drive/MyDrive/Project Portofolio/"
    "Medical Appointment No Shows/data/Raw.csv"
)

# Lokasi output dataset bersih
OUTPUT_DATA_PATH = Path(
    "/content/drive/MyDrive/Project Portofolio/"
    "Medical Appointment No Shows/data/"
    "Processed.csv"
)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"File dataset tidak ditemukan: {RAW_DATA_PATH}\n"
        "Silakan periksa kembali nilai RAW_DATA_PATH."
    )

df_raw = pd.read_csv(RAW_DATA_PATH)
df_clean = df_raw.copy()

print(f"Jumlah baris: {df_clean.shape[0]:,}")
print(f"Jumlah kolom: {df_clean.shape[1]}")


Jumlah baris: 110,527
Jumlah kolom: 14


In [32]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Pemeriksaan Awal Dataset

In [33]:

# Melihat lima baris pertama
display(df_clean.head())


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
0,29872499824296.00,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,558997776694438.00,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4262962299951.00,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,867951213174.00,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8841186448183.00,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No


In [34]:

# Melihat lima baris terakhir
display(df_clean.tail())


,PatientId,AppointmentID,Gender,ScheduledDay,AppointmentDay,Age,Neighbourhood,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received,No-show
110522,2572134369293.00,5651768,F,2016-05-03T09:15:35Z,2016-06-07T00:00:00Z,56,MARIA ORTIZ,0,0,0,0,0,1,No
110523,3596266328735.00,5650093,F,2016-05-03T07:27:33Z,2016-06-07T00:00:00Z,51,MARIA ORTIZ,0,0,0,0,0,1,No
110524,15576631729893.00,5630692,F,2016-04-27T16:03:52Z,2016-06-07T00:00:00Z,21,MARIA ORTIZ,0,0,0,0,0,1,No
110525,92134931435557.00,5630323,F,2016-04-27T15:09:23Z,2016-06-07T00:00:00Z,38,MARIA ORTIZ,0,0,0,0,0,1,No
110526,377511518121127.00,5629448,F,2016-04-27T13:30:56Z,2016-06-07T00:00:00Z,54,MARIA ORTIZ,0,0,0,0,0,1,No


In [35]:

# Informasi struktur, tipe data, dan jumlah non-null
df_clean.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 110527 entries, 0 to 110526
Data columns (total 14 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   PatientId       110527 non-null  float64
 1   AppointmentID   110527 non-null  int64  
 2   Gender          110527 non-null  object 
 3   ScheduledDay    110527 non-null  object 
 4   AppointmentDay  110527 non-null  object 
 5   Age             110527 non-null  int64  
 6   Neighbourhood   110527 non-null  object 
 7   Scholarship     110527 non-null  int64  
 8   Hipertension    110527 non-null  int64  
 9   Diabetes        110527 non-null  int64  
 10  Alcoholism      110527 non-null  int64  
 11  Handcap         110527 non-null  int64  
 12  SMS_received    110527 non-null  int64  
 13  No-show         110527 non-null  object 
dtypes: float64(1), int64(8), object(5)
memory usage: 11.8+ MB


In [36]:

# Statistik deskriptif untuk kolom numerik
display(df_clean.describe())


,PatientId,AppointmentID,Age,Scholarship,Hipertension,Diabetes,Alcoholism,Handcap,SMS_received
count,110527.00,110527.00,110527.00,110527.00,110527.00,110527.00,110527.00,110527.00,110527.00
mean,147496265710394.06,5675305.12,37.09,0.10,0.20,0.07,0.03,0.02,0.32
std,256094920291738.88,71295.75,23.11,0.30,0.40,0.26,0.17,0.16,0.47
min,39217.84,5030230.00,-1.00,0.00,0.00,0.00,0.00,0.00,0.00
25%,4172614444192.00,5640285.50,18.00,0.00,0.00,0.00,0.00,0.00,0.00
50%,31731838713978.00,5680573.00,37.00,0.00,0.00,0.00,0.00,0.00,0.00
75%,94391720898175.00,5725523.50,55.00,0.00,0.00,0.00,0.00,0.00,1.00
max,999981631772427.00,5790484.00,115.00,1.00,1.00,1.00,1.00,4.00,1.00



## 4. Pemeriksaan Missing Values

Missing values perlu diperiksa sebelum transformasi karena dapat memengaruhi perhitungan statistik dan pembuatan fitur baru.


In [37]:

missing_summary = (
    pd.DataFrame({
        "missing_count": df_clean.isna().sum(),
        "missing_percentage": (df_clean.isna().mean() * 100).round(2)
    })
    .query("missing_count > 0")
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)


,missing_count,missing_percentage



## 5. Pemeriksaan Duplikasi

`AppointmentID` merepresentasikan identitas janji temu, sehingga perlu diperiksa apakah terdapat nilai duplikat. Sementara itu, `PatientId` boleh muncul lebih dari sekali karena satu pasien dapat memiliki beberapa janji temu.


In [38]:

duplicate_rows = df_clean.duplicated().sum()
duplicate_appointment_id = df_clean["AppointmentID"].duplicated().sum()
duplicate_patient_id = df_clean["PatientId"].duplicated().sum()

print(f"Duplikasi seluruh baris       : {duplicate_rows:,}")
print(f"Duplikasi AppointmentID       : {duplicate_appointment_id:,}")
print(f"Duplikasi PatientId           : {duplicate_patient_id:,}")


Duplikasi seluruh baris       : 0
Duplikasi AppointmentID       : 0
Duplikasi PatientId           : 48,228


## 6. Menstandarkan Nama Kolom

In [39]:

# Menggunakan snake_case agar nama kolom konsisten dan mudah digunakan
column_mapping = {
    "PatientId": "patient_id",
    "AppointmentID": "appointment_id",
    "Gender": "gender",
    "ScheduledDay": "scheduled_day",
    "AppointmentDay": "appointment_day",
    "Age": "age",
    "Neighbourhood": "neighbourhood",
    "Scholarship": "scholarship",
    "Hipertension": "hypertension",
    "Diabetes": "diabetes",
    "Alcoholism": "alcoholism",
    "Handcap": "handicap",
    "SMS_received": "sms_received",
    "No-show": "no_show",
}

df_clean = df_clean.rename(columns=column_mapping)

display(df_clean.head())


,patient_id,appointment_id,gender,scheduled_day,appointment_day,age,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap,sms_received,no_show
0,29872499824296.00,5642903,F,2016-04-29T18:38:08Z,2016-04-29T00:00:00Z,62,JARDIM DA PENHA,0,1,0,0,0,0,No
1,558997776694438.00,5642503,M,2016-04-29T16:08:27Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,0,0,0,0,0,No
2,4262962299951.00,5642549,F,2016-04-29T16:19:04Z,2016-04-29T00:00:00Z,62,MATA DA PRAIA,0,0,0,0,0,0,No
3,867951213174.00,5642828,F,2016-04-29T17:29:31Z,2016-04-29T00:00:00Z,8,PONTAL DE CAMBURI,0,0,0,0,0,0,No
4,8841186448183.00,5642494,F,2016-04-29T16:07:23Z,2016-04-29T00:00:00Z,56,JARDIM DA PENHA,0,1,1,0,0,0,No



## 7. Konversi Kolom Tanggal

Kolom `scheduled_day` dan `appointment_day` dikonversi ke tipe `datetime`.

- `scheduled_day`: waktu ketika janji temu dibuat.
- `appointment_day`: tanggal ketika pasien dijadwalkan untuk datang.

`errors="coerce"` digunakan agar nilai tanggal yang tidak dapat diproses menjadi `NaT`, sehingga dapat diperiksa secara eksplisit.


In [40]:

date_columns = ["scheduled_day", "appointment_day"]

for column in date_columns:
    df_clean[column] = pd.to_datetime(
        df_clean[column],
        errors="coerce",
        utc=True
    )

print(df_clean[date_columns].dtypes)
print("\nJumlah tanggal tidak valid:")
print(df_clean[date_columns].isna().sum())


scheduled_day      datetime64[ns, UTC]
appointment_day    datetime64[ns, UTC]
dtype: object

Jumlah tanggal tidak valid:
scheduled_day      0
appointment_day    0
dtype: int64



## 8. Validasi dan Penanganan Nilai Usia

Pada dataset ini, usia `0` atau nilai negatif tidak dianggap sebagai usia pasien yang valid. Nilai di atas `100` juga ditandai sebagai tidak valid untuk kebutuhan analisis.

Nilai usia yang tidak valid diubah menjadi `NaN`, bukan menghapus seluruh baris, karena informasi lain pada baris tersebut masih dapat berguna.


In [41]:

# Menandai usia yang tidak valid.
# Missing value tidak dimasukkan ke dalam flag invalid karena merupakan kasus terpisah.
df_clean["age_invalid"] = (
    df_clean["age"].notna()
    & ~df_clean["age"].between(1, 100)
)

print(f"Jumlah usia tidak valid: {df_clean['age_invalid'].sum():,}")

display(
    df_clean.loc[df_clean["age_invalid"], ["patient_id", "appointment_id", "age"]]
)


Jumlah usia tidak valid: 3,547


,patient_id,appointment_id,age
59,71844284745331.00,5638545,0
63,236623344873175.00,5628286,0
64,188517384712787.00,5616082,0
65,271881817799985.00,5628321,0
67,86471282513499.00,5639264,0
...,...,...,...
110345,147395196662956.00,5702537,0
110346,5577525313231.00,5777724,0
110454,614245995575.00,5772400,0
110460,43218463343323.00,5769545,0


In [42]:

# Mengubah usia tidak valid menjadi missing value
df_clean.loc[df_clean["age_invalid"], "age"] = np.nan

print(f"Jumlah missing age setelah cleaning: {df_clean['age'].isna().sum():,}")
display(df_clean["age"].describe())


Jumlah missing age setelah cleaning: 3,547


,age
count,106980.00
mean,38.31
std,22.46
min,1.00
25%,19.00
50%,38.00
75%,56.00
max,100.00



## 9. Membuat Kelompok Usia

Fitur `age_group` dibuat agar distribusi pasien lebih mudah dianalisis di SQL atau Tableau.

Pasien yang usia-nya tidak tersedia akan dimasukkan ke kategori `Unknown`.


In [43]:

age_bins = [0, 12, 18, 35, 50, 65, 75, 100]
age_labels = [
    "0-12",
    "13-18",
    "19-35",
    "36-50",
    "51-65",
    "66-75",
    "76+",
]

df_clean["age_group"] = pd.cut(
    df_clean["age"],
    bins=age_bins,
    labels=age_labels,
    include_lowest=True
).astype("object")

df_clean["age_group"] = df_clean["age_group"].fillna("Unknown")

display(
    df_clean["age_group"]
    .value_counts(dropna=False)
    .rename_axis("age_group")
    .reset_index(name="count")
)


,age_group,count
0,19-35,24137
1,51-65,22122
2,36-50,22100
3,0-12,17497
4,66-75,7909
5,13-18,7830
6,76+,5385
7,Unknown,3547



## 10. Menghitung `waiting_days`

`waiting_days` dihitung sebagai selisih hari antara `appointment_day` dan `scheduled_day`.

Tanggal dinormalisasi terlebih dahulu menggunakan `.dt.normalize()` agar perbedaan jam tidak memengaruhi hasil perhitungan.


In [44]:

scheduled_date = df_clean["scheduled_day"].dt.normalize()
appointment_date = df_clean["appointment_day"].dt.normalize()

df_clean["waiting_days"] = (
    appointment_date - scheduled_date
).dt.days

display(
    df_clean[
        ["scheduled_day", "appointment_day", "waiting_days"]
    ].head()
)


,scheduled_day,appointment_day,waiting_days
0,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,0
1,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,0
2,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,0
3,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,0
4,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,0



### 10.1. Menangani `waiting_days` Negatif

Nilai negatif menunjukkan bahwa `appointment_day` lebih awal daripada `scheduled_day`. Kondisi ini tidak sesuai dengan interpretasi waktu tunggu dan diperlakukan sebagai nilai tidak valid.

Nilai negatif diubah menjadi `NaN` agar tidak digunakan dalam analisis waktu tunggu.


In [45]:

df_clean["waiting_days_invalid"] = df_clean["waiting_days"] < 0

negative_waiting = df_clean[df_clean["waiting_days_invalid"]]

print(f"Jumlah waiting_days negatif: {len(negative_waiting):,}")

display(
    negative_waiting[
        [
            "patient_id",
            "appointment_id",
            "scheduled_day",
            "appointment_day",
            "waiting_days",
            "no_show",
        ]
    ].head(10)
)


Jumlah waiting_days negatif: 5


,patient_id,appointment_id,scheduled_day,appointment_day,waiting_days,no_show
27033,7839272661752.00,5679978,2016-05-10 10:51:53+00:00,2016-05-09 00:00:00+00:00,-1,Yes
55226,7896293967868.00,5715660,2016-05-18 14:50:41+00:00,2016-05-17 00:00:00+00:00,-1,Yes
64175,24252258389979.00,5664962,2016-05-05 13:43:58+00:00,2016-05-04 00:00:00+00:00,-1,Yes
71533,998231581612122.00,5686628,2016-05-11 13:49:20+00:00,2016-05-05 00:00:00+00:00,-6,Yes
72362,3787481966821.00,5655637,2016-05-04 06:50:57+00:00,2016-05-03 00:00:00+00:00,-1,Yes


In [46]:

df_clean.loc[df_clean["waiting_days_invalid"], "waiting_days"] = np.nan

print(f"Jumlah missing waiting_days: {df_clean['waiting_days'].isna().sum():,}")
display(df_clean["waiting_days"].describe())


Jumlah missing waiting_days: 5


,waiting_days
count,110522.00
mean,10.18
std,15.26
min,0.00
25%,0.00
50%,4.00
75%,15.00
max,179.00



## 11. Pemeriksaan Nilai Kategorikal dan Biner

Pemeriksaan ini digunakan untuk memastikan nilai kategorikal tidak mengandung ejaan atau kategori yang tidak diharapkan.


In [47]:

categorical_columns = [
    "gender",
    "neighbourhood",
    "no_show",
]

for column in categorical_columns:
    print(f"\n--- {column} ---")
    display(df_clean[column].value_counts(dropna=False).head(20))



--- gender ---


,count
gender,
F,71840
M,38687



--- neighbourhood ---


,count
neighbourhood,
JARDIM CAMBURI,7717
MARIA ORTIZ,5805
RESISTÊNCIA,4431
JARDIM DA PENHA,3877
ITARARÉ,3514
CENTRO,3334
TABUAZEIRO,3132
SANTA MARTHA,3131
JESUS DE NAZARETH,2853



--- no_show ---


,count
no_show,
No,88208
Yes,22319


In [48]:

binary_columns = [
    "scholarship",
    "hypertension",
    "diabetes",
    "alcoholism",
    "handicap",
    "sms_received",
]

for column in binary_columns:
    print(f"\n--- {column} ---")
    display(df_clean[column].value_counts(dropna=False))



--- scholarship ---


,count
scholarship,
0,99666
1,10861



--- hypertension ---


,count
hypertension,
0,88726
1,21801



--- diabetes ---


...

,count
alcoholism,
0,107167
1,3360



--- handicap ---


,count
handicap,
0,108286
1,2042
2,183
3,13
4,3



--- sms_received ---


,count
sms_received,
0,75045
1,35482



## 12. Membuat Variabel Target Biner

Kolom `no_show` memiliki nilai:

- `No`: pasien hadir.
- `Yes`: pasien tidak hadir.

Untuk memudahkan analisis agregasi dan visualisasi, dibuat kolom `no_show_binary` dengan aturan:

- `No` → `0`
- `Yes` → `1`


In [49]:

df_clean["no_show_binary"] = (
    df_clean["no_show"]
    .map({"No": 0, "Yes": 1})
    .astype("Int64")
)

display(
    df_clean[["no_show", "no_show_binary"]]
    .drop_duplicates()
    .sort_values("no_show_binary")
)


,no_show,no_show_binary
0,No,0
6,Yes,1



## 13. Menyiapkan Dataset Final

Kolom flag seperti `age_invalid` dan `waiting_days_invalid` digunakan selama proses audit cleaning, tetapi tidak harus dibawa ke dataset analisis utama.

Dataset final berisi kolom identitas, atribut pasien, informasi janji temu, fitur analisis, dan target biner.


In [50]:

final_columns = [
    "patient_id",
    "appointment_id",
    "gender",
    "scheduled_day",
    "appointment_day",
    "age",
    "age_group",
    "neighbourhood",
    "scholarship",
    "hypertension",
    "diabetes",
    "alcoholism",
    "handicap",
    "sms_received",
    "no_show",
    "no_show_binary",
    "waiting_days",
]

df_final = df_clean[final_columns].copy()

# Menyesuaikan tipe data numerik yang masih dapat memiliki missing value
nullable_integer_columns = [
    "patient_id",
    "appointment_id",
    "age",
    "waiting_days",
]

for column in nullable_integer_columns:
    df_final[column] = df_final[column].round().astype("Int64")

display(df_final.head())


,patient_id,appointment_id,gender,scheduled_day,appointment_day,age,age_group,neighbourhood,scholarship,hypertension,diabetes,alcoholism,handicap,sms_received,no_show,no_show_binary,waiting_days
0,29872499824296,5642903,F,2016-04-29 18:38:08+00:00,2016-04-29 00:00:00+00:00,62,51-65,JARDIM DA PENHA,0,1,0,0,0,0,No,0,0
1,558997776694438,5642503,M,2016-04-29 16:08:27+00:00,2016-04-29 00:00:00+00:00,56,51-65,JARDIM DA PENHA,0,0,0,0,0,0,No,0,0
2,4262962299951,5642549,F,2016-04-29 16:19:04+00:00,2016-04-29 00:00:00+00:00,62,51-65,MATA DA PRAIA,0,0,0,0,0,0,No,0,0
3,867951213174,5642828,F,2016-04-29 17:29:31+00:00,2016-04-29 00:00:00+00:00,8,0-12,PONTAL DE CAMBURI,0,0,0,0,0,0,No,0,0
4,8841186448183,5642494,F,2016-04-29 16:07:23+00:00,2016-04-29 00:00:00+00:00,56,51-65,JARDIM DA PENHA,0,1,1,0,0,0,No,0,0


## 14. Validasi Akhir

In [51]:

print("Ukuran dataset final:", df_final.shape)

print(
    "Duplikasi appointment_id:",
    df_final["appointment_id"].duplicated().sum()
)

print("\nMissing values:")
display(
    df_final.isna()
    .sum()
    .sort_values(ascending=False)
    .to_frame("missing_count")
)

print("\nDistribusi no_show_binary:")
display(df_final["no_show_binary"].value_counts(dropna=False))

print(
    "\nWaiting days negatif:",
    (df_final["waiting_days"] < 0).sum()
)


Ukuran dataset final: (110527, 17)
Duplikasi appointment_id: 0

Missing values:


,missing_count
age,3547
waiting_days,5
patient_id,0
scheduled_day,0
appointment_id,0
appointment_day,0
age_group,0
neighbourhood,0
gender,0
scholarship,0



Distribusi no_show_binary:


,count
no_show_binary,
0,88208
1,22319



Waiting days negatif: 0



## 15. Mengekspor Dataset Bersih

File CSV berikut akan digunakan sebagai input untuk tahap berikutnya, yaitu:

1. **SQL:** eksplorasi, agregasi, dan pembuatan tabel analitik.
2. **Tableau:** pembuatan dashboard dan visualisasi interaktif.


In [52]:

OUTPUT_DATA_PATH.parent.mkdir(parents=True, exist_ok=True)

df_final.to_csv(
    OUTPUT_DATA_PATH,
    index=False
)

print(f"Dataset final berhasil disimpan di: {OUTPUT_DATA_PATH}")
print(f"Ukuran dataset yang disimpan: {df_final.shape}")


Dataset final berhasil disimpan di: /content/drive/MyDrive/Project Portofolio/Medical Appointment No Shows/data/Processed.csv
Ukuran dataset yang disimpan: (110527, 17)



## Kesimpulan Tahap Data Cleaning

Pada tahap ini, dataset telah:

- dimuat dan diperiksa struktur awalnya;
- menggunakan nama kolom yang konsisten;
- dikonversi ke format tanggal;
- diperiksa missing values dan duplikasi;
- menangani usia yang tidak valid;
- menghitung `waiting_days`;
- mengubah `waiting_days` negatif menjadi missing value;
- membuat `age_group` dan `no_show_binary`;
- divalidasi dan diekspor sebagai CSV bersih.

> **Catatan untuk portofolio:** Hasil cleaning sebaiknya dijelaskan bersama alasan bisnis atau analitis di balik setiap keputusan, bukan hanya menampilkan kode.
